# Day 1 — Maryam's Local Version

**Fully local, no API key needed.**

This notebook does everything `day1.ipynb` does — scrape a URL and summarize it with an LLM — but uses **Ollama** running on your machine instead of OpenAI.

### Prerequisites
1. Install Ollama: `brew install ollama`
2. Start the server: `ollama serve` (in a separate terminal)
3. Pull the model: `ollama pull llama3.2`
4. Make sure your `udemy_env` conda environment is active and selected as the kernel

In [ ]:
# Verify Ollama is reachable before doing anything else
import requests

try:
    r = requests.get("http://localhost:11434")
    print(r.text)  # should print: Ollama is running
except Exception as e:
    print(f"Ollama not reachable: {e}")
    print("Run 'ollama serve' in a terminal first.")

In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display
from scraper import fetch_website_contents

In [ ]:
# Point the OpenAI client library at local Ollama — no real API key needed
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODEL = "llama3.2"

In [ ]:
# Quick sanity check — say hello to the local model
response = ollama.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello! Reply in one sentence."}]
)
print(response.choices[0].message.content)

## Prompts

We give the model two messages every time:
- **system** — tells it what role to play
- **user** — the actual scraped text to summarize

In [ ]:
system_prompt = """
You are an assistant that analyzes the contents of a website
and provides a short, clear summary, ignoring navigation text.
Respond in markdown.
"""

In [ ]:
def messages_for(url_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": (
            "Here are the contents of a website.\n"
            "Provide a short summary. If it includes news or announcements, summarize those too.\n\n"
            + url_text
        )}
    ]

## Summarization pipeline

```
URL  →  scraper.py (BeautifulSoup)  →  raw text  →  llama3.2 (local)  →  markdown summary
```

In [ ]:
def summarize(url):
    text = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(text)
    )
    return response.choices[0].message.content

def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary("https://edwarddonner.com")

In [ ]:
# Try any other URL you like
display_summary("https://anthropic.com")